# Week 3 — Imbalanced Classification & LightGBM Modeling
## Member 3 — Context & Integration

## 1. Environment Setup & Imports
## 2. Load Fused Dataset
## 3. Feature Matrix Extraction
## 4. NaN Verification
## 5. Class Distribution

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder

print("All imports successful")

## 2. Load Fused Dataset

In [ ]:
df = pd.read_csv('../data/ai4i2020.csv')

np.random.seed(42)
df['ambient_temp_C'] = np.random.normal(loc=28, scale=5, size=len(df))
df['factory_load_pct'] = np.random.uniform(50, 100, size=len(df))
df['humidity_pct'] = np.random.normal(loc=60, scale=10, size=len(df))

le = LabelEncoder()
df['Type_enc'] = le.fit_transform(df['Type'])

print("Fused dataset loaded!")
print("Shape:", df.shape)

In [ ]:
ext_features = [
    'Air temperature [K]', 'Process temperature [K]',
    'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]',
    'Type_enc', 'ambient_temp_C', 'factory_load_pct', 'humidity_pct'
]

X = df[ext_features]
y = df['Machine failure']

print("Feature Matrix X shape:", X.shape)
print("Target y shape:", y.shape)

In [ ]:
print("NaN check:")
print(X.isnull().sum().sum())

if X.isnull().sum().sum() == 0:
    print("✅ No NaN values — dataset is clean!")

In [ ]:
print("Class Distribution:")
print(y.value_counts())
print(f"\nNo Failure: {(y==0).sum()} — {(y==0).mean()*100:.2f}%")
print(f"Failure: {(y==1).sum()} — {(y==1).mean()*100:.2f}%")
print(f"Imbalance Ratio: {(y==0).sum()/(y==1).sum():.1f}:1")
print("\n✅ Dataset ready for SMOTE + LightGBM modeling!")

## Commentary — Feature Matrix & Class Distribution

- Total features: 9 (5 internal sensors + Type_enc + 3 external context)
- Total samples: 10,000
- No NaN values — dataset clean and ready
- Imbalance Ratio: 28.5:1 — SMOTE will be applied inside training folds only

## 6. SMOTE Validation — No Data Leakage Check

In [ ]:
from sklearn.model_selection import StratifiedKFold
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from lightgbm import LGBMClassifier

# Setup 5-fold stratified cross validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Test SMOTE on first fold only to validate
for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    print(f"=== Fold {fold+1} ===")
    print(f"Training set size: {len(X_train)}")
    print(f"Test set size: {len(X_test)}")
    print(f"\nClass counts BEFORE SMOTE (training only):")
    print(y_train.value_counts())
    
    # Apply SMOTE only on training data
    smote = SMOTE(random_state=42)
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)
    
    print(f"\nClass counts AFTER SMOTE (training only):")
    import pandas as pd
    print(pd.Series(y_train_resampled).value_counts())
    
    print(f"\nTest set class counts (UNCHANGED — no leakage):")
    print(y_test.value_counts())
    
    # Assert SMOTE only applied to training data
    assert len(X_test) == len(y_test), "Test set size mismatch!"
    assert y_test.value_counts()[1] < y_test.value_counts()[0], "Test set still imbalanced — no leakage confirmed!"
    
    print(f"\n✅ Fold {fold+1}: SMOTE applied correctly — NO DATA LEAKAGE!")
    break  # Only test first fold for validation

## Commentary — SMOTE Validation

SMOTE validation confirms:

- **SMOTE is applied ONLY on training data** inside each fold — never on test data
- **Before SMOTE:** Training set is imbalanced (28.5:1 ratio)
- **After SMOTE:** Training set is balanced (1:1 ratio) — equal failure and no-failure samples
- **Test set remains unchanged** — still imbalanced, reflecting real-world distribution
- **No data leakage confirmed** — future test samples never influence the training process

This is critical for getting honest model evaluation metrics. A common mistake is applying SMOTE before splitting — this would let synthetic samples "leak" information about the test set into training, inflating performance metrics artificially.